# OceanGuard AI · Gemma 4 Fine-Tune for Marine Debris Detection (Unsloth Bonus)

**Reproducibility companion notebook · Unsloth $10K bonus track · Kaggle Gemma 4 Good Hackathon**

Author: Alejandro Sanchez Ferrer (`asanc.tech@gmail.com`)

Source repo: https://github.com/asferrer/OceanguardAI-App

HuggingFace adapter: https://huggingface.co/asferrer/gemma-4-E2B-it-oceanguard-marine-debris

License: Apache 2.0 (LoRA adapter) · Gemma Terms of Use (base model)

---

**Abstract.** This notebook reproduces the open-vocabulary marine debris detection fine-tune
of `google/gemma-4-E2B-it` performed for the OceanGuard AI Android application. The training
uses Unsloth FastVisionModel in a two-stage LoRA recipe (text-side first, then vision-side
unfrozen with a VQA mix) on a merged COCO dataset (CleanSea + Ocean_garbage + Neural_Ocean)
covering 50 marine debris classes mapped onto 11 ecological-impact families. The published
adapter is consumed at inference time through the same `DETECTION_PROMPT` shipped in the
Android app, producing the `box_2d` + `label` + `material` JSON contract that the on-device
pipeline parses.

<!-- OG_KAGGLE_SETUP_BANNER -->
## Setup on Kaggle — required before running

This fine-tune notebook **must** run on a Kaggle GPU runtime — Unsloth's `FastVisionModel` and the 4-bit
quantised Gemma 4 base require CUDA. Before clicking **Run All**:

1. Open the **Settings** panel on the right of the notebook editor.
2. **Accelerator** → choose **GPU T4 x2** (recommended; 2 × 16 GiB) or **GPU P100** (single 16 GiB).
   CPU mode will fail at the `FastVisionModel.from_pretrained` step.
3. **Internet** → **On**. Required to download the base Gemma 4 weights and the published OceanGuard adapter from Hugging Face.
   - Internet on Kaggle requires a one-time **phone verification** on your account (Settings → Account).
4. **Persistence** → **Files only**, so the multi-hour fine-tune can resume from disk if the session restarts.
5. (Optional, for the full Stage-1 + Stage-2 run) attach the dataset version of the marine-debris COCO corpus as an input dataset; the demo cells below run on a small synthetic sample so the rest of the notebook is reproducible end-to-end inside the 9-hour Kaggle session limit.

GPU quota on the free tier is ~30 hours / week. A full LoRA Stage-1 + Stage-2 pass on T4 x2 takes ~6–8 h; the demo path in this notebook uses < 10 minutes.


## 1. Motivation — Why fine-tune Gemma 4 for marine debris?

The shipped APK uses **base** `google/gemma-4-E2B-it` for single-shot deep analysis. Base
Gemma 4 already produces usable `box_2d` JSON on the open-vocabulary 50-class taxonomy, but
three measurable failure modes motivate a domain fine-tune:

1. **Taxonomy drift.** Base Gemma 4 emits labels like `bottle`, `trash`, `floating object`
   that the Kotlin resolver in `Gemma4VisionDetector.LABEL_TO_TYPE` then has to coerce into
   one of 50 `DebrisType` enum values. ~12% of detections during pilot fall to the
   `PLASTIC_DEBRIS` catch-all and lose material attribution.
2. **JSON-validity regressions.** Under prompt budget pressure (~200 tokens of system +
   ~150 tokens of detection prompt) the model occasionally emits Markdown fences, mid-string
   line breaks, or partial arrays. The notebook tracks `json_valid_rate` as a primary metric.
3. **Material confabulation.** Without explicit grounding, base Gemma 4 sometimes assigns
   `plastic` to glass bottles or `metal` to plastic fishing buoys, which propagates into the
   ecological-impact family lookup and biases downstream reports.

The two-stage Unsloth recipe targets these three failure modes specifically: stage 1 hammers
JSON contract and label taxonomy on the text head; stage 2 unfreezes the vision tower with a
10% VQA-mix to recover material attribution without catastrophic forgetting on the general
visual prior.

**Deployment status.** The fine-tune currently lives in this notebook and on HuggingFace.
Shipping a fine-tuned `.litertlm` to the Android APK is blocked on the LiteRT-LM checkpoint
conversion path (Google AI Edge `ai-edge-torch` PEFT-export support is in progress upstream
at the time of submission, May 2026). The path forward is documented in section 12.

In [ ]:
# Unsloth pulls a CUDA-compatible torch wheel; pin transformers via Unsloth's own constraints.
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q --upgrade pycocotools pillow matplotlib pandas numpy

In [ ]:
import json, os, re, io, sys, time, math, base64, urllib.request
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# OG_KAGGLE_GPU_DIAG
# Hard requirement: CUDA must be available. Unsloth FastVisionModel + 4-bit
# bitsandbytes need a real CUDA GPU.
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. On Kaggle: Settings → Accelerator → GPU T4 x2. "
        "Unsloth FastVisionModel cannot run on CPU."
    )
dev = torch.cuda.current_device()
props = torch.cuda.get_device_properties(dev)
print(f"Torch     : {torch.__version__}")
print(f"CUDA avail: True ({torch.version.cuda})")
print(f"GPU       : {torch.cuda.get_device_name(dev)}")
print(f"VRAM      : {props.total_memory / 1024**3:.1f} GiB total")
print(f"SM count  : {props.multi_processor_count}")
print(f"GPUs seen : {torch.cuda.device_count()}  (T4 x2 reports 2)")


## 2. Load base Gemma 4 E2B via Unsloth FastVisionModel

Unsloth's `FastVisionModel` wraps the HuggingFace vision-language head with 4-bit NF4
quantization and patched attention, which is what makes a Gemma 4 E2B fine-tune fit in
a Kaggle T4 (16 GiB). On the original training host (RTX 5090, 32 GiB) we use the same
API with `load_in_4bit=False` for full bf16 training; everything else in the notebook is
identical.

In [ ]:
from unsloth import FastVisionModel

BASE_MODEL = "unsloth/gemma-4-E2B-it"  # mirror of google/gemma-4-E2B-it with Unsloth chat-template patches
ADAPTER_REPO = "asferrer/gemma-4-E2B-it-oceanguard-marine-debris"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = BASE_MODEL,
    load_in_4bit = True,        # T4-friendly; flip to False on RTX 5090 for full bf16
    use_gradient_checkpointing = "unsloth",  # ~30% memory saving at training time
)

FastVisionModel.for_inference(model)  # disable training-only optimisations for the demo
print(f"Base model loaded: {BASE_MODEL}")
print(f"Params (total)  : ~5.1B  |  Active (E2B sparse path): ~2.3B")

## 3. Load published OceanGuard LoRA adapter

The fine-tuned weights are distributed as a PEFT LoRA adapter on HuggingFace. Loading the
adapter is a small (`~50 MB`) download on top of the 4-bit base, which makes it cheap to
experiment with the OceanGuard fine-tune without re-training. The next cell is wrapped in a
try/except so the notebook is still **Run-All clean** if the adapter has not been published
yet at the time of execution — in that case the demo cells run on the base model only.

In [ ]:
ADAPTER_LOADED = False
try:
    # Try the modern PEFT path first; it tolerates base-model name drift between the
    # Unsloth FP16 mirror loaded above and the bnb-4bit base recorded in adapter_config.json.
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_REPO, adapter_name="oceanguard")
    model.set_adapter("oceanguard")
    ADAPTER_LOADED = True
    print(f"[OK] OceanGuard LoRA adapter active ({ADAPTER_REPO}).")
except Exception as exc:
    # Fall back to PreTrainedModel.load_adapter (transformers >= 4.41).
    try:
        model.load_adapter(ADAPTER_REPO, adapter_name="oceanguard")
        model.set_adapter("oceanguard")
        ADAPTER_LOADED = True
        print(f"[OK] OceanGuard LoRA adapter active ({ADAPTER_REPO}) [via load_adapter].")
    except Exception as exc2:
        print(f"[INFO] Adapter not loaded ({type(exc2).__name__}: {exc2}).")
        print("        Falling back to BASE model for the rest of the notebook.")


## 4. Inference helper — the exact `DETECTION_PROMPT` shipped on Android

The Android detector in `Gemma4VisionDetector.kt` uses a short open-vocabulary prompt designed
to bias the model toward `*_debris` material fallbacks when the specific subtype is ambiguous.
We reuse the **identical** prompt here so the notebook is a faithful reproduction of the on-device
behaviour, modulo runtime (Transformers vs. LiteRT-LM).

In [ ]:
# Verbatim from android/app/src/main/java/com/oceanguard/inference/Gemma4VisionDetector.kt
DETECTION_PROMPT = """Detect ALL marine debris and litter in this image. Output ONLY a JSON array.

For each object, provide:
- "box_2d": bounding box as [y_min, x_min, y_max, x_max], integers 0-1000
- "label": specific object type using snake_case
  (e.g. plastic_bottle, glass_bottle, aluminum_can, fishing_net, fishing_line, rope, cigarette_butt, plastic_bag, styrofoam, glove, mask, tire, syringe, battery, clothing, cardboard, lumber, ceramic_fragment, paint_can)
- "material": the primary material (Plastic, Metal, Glass, Rubber, Fabric, Fishing_Net, Wood, Paper, Ceramic, Chemical)

Output format:
[{"box_2d": [y_min, x_min, y_max, x_max], "label": "object_type", "material": "material_type"}]

Be specific: distinguish plastic_bottle from glass_bottle, aluminum_can from tin_can, etc.
If you can't identify the specific type, fall back to a material-based label:
glass_debris, metal_debris, plastic_debris, fabric_debris (one of these is always correct).
If no debris is found, output: []"""

SYSTEM_MESSAGE = (
    "You are a marine debris detection system. Output ONLY valid JSON arrays. "
    "Never include explanations, markdown, or text outside the JSON."
)

_JSON_ARRAY_RE = re.compile(r"\[.*\]", re.DOTALL)

def extract_json_array(raw: str):
    """Mirror of Gemma4VisionDetector.extractJsonArray — tolerant of code fences."""
    raw = raw.strip()
    fence = re.search(r"```(?:json)?\s*\n?(\[.*?\])\s*```", raw, re.DOTALL)
    if fence:
        return fence.group(1)
    match = _JSON_ARRAY_RE.search(raw)
    return match.group(0) if match else None


def detect_debris(image_path_or_pil, max_new_tokens: int = 1024, temperature: float = 0.3):
    """Run a single Gemma 4 vision detection pass and return (parsed_detections, raw_text, elapsed_s).

    Returns a 3-tuple so callers can distinguish JSON-invalid generations (parsed is None) from
    legitimate empty surveys (parsed is []).
    """
    if isinstance(image_path_or_pil, (str, Path)):
        image = Image.open(image_path_or_pil).convert("RGB")
    else:
        image = image_path_or_pil.convert("RGB")

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_MESSAGE}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": DETECTION_PROMPT},
        ]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    t0 = time.time()
    with torch.inference_mode():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0.0,
            temperature=temperature,
            top_p=0.95,
        )
    elapsed = time.time() - t0

    # Strip the prompt tokens before decoding so we don't waste post-processing on the system message.
    new_ids = gen_ids[0][inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(new_ids, skip_special_tokens=True)

    json_blob = extract_json_array(raw_text)
    if json_blob is None:
        return None, raw_text, elapsed
    try:
        parsed = json.loads(json_blob)
    except json.JSONDecodeError:
        return None, raw_text, elapsed
    return parsed, raw_text, elapsed

## 5. Demo — base vs. fine-tuned on representative images

The five demo images below come from the held-out evaluation slice of the merged COCO dataset.
If the notebook is running on Kaggle and the `oceanguard-marine-debris-demo` dataset has been
attached, they are loaded from disk; otherwise we fall back to a tiny set of public-domain
Wikimedia thumbnails so the cell is always runnable.

Each image is rendered with bounding boxes drawn from the parsed `box_2d` output. We then
tabulate the qualitative differences between the base model and the fine-tuned adapter.

In [ ]:
DEMO_DIR = Path("/kaggle/input/oceanguard-marine-debris-demo") if Path("/kaggle/input").exists() else Path("demo_images")
DEMO_DIR.mkdir(parents=True, exist_ok=True)

# Wikimedia Commons rejects requests without a meaningful User-Agent (HTTP 400).
# https://meta.wikimedia.org/wiki/User-Agent_policy
WIKIMEDIA_UA = "OceanGuardAI-Notebook/1.0 (https://github.com/asferrer/OceanguardAI-App; asanc.tech@gmail.com)"

# Public-domain / CC-BY thumbnails verified 2026-05 (HTTP 200 with proper UA).
# Replace with an attached Kaggle dataset for the real eval slice.
FALLBACK_URLS = [
    ("plastic_pollution_ghana.jpg",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/9/9a/Plastic_Pollution_in_Ghana.jpg/960px-Plastic_Pollution_in_Ghana.jpg"),
    ("turtle_ghost_net.jpg",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/5/50/Turtle_entangled_in_marine_debris_%28ghost_net%29.jpg/960px-Turtle_entangled_in_marine_debris_%28ghost_net%29.jpg"),
    ("beach_litter_scabbacombe.jpg",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/Collected_Beach_Litter_at_Scabbacombe_Sands.jpg/960px-Collected_Beach_Litter_at_Scabbacombe_Sands.jpg"),
    ("beach_pollution_tenerife.jpg",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/7/78/004_Beach_pollution_in_Tenerife_-_Atlantic_Ocean_beach_plastic_garbage_debris.jpg/960px-004_Beach_pollution_in_Tenerife_-_Atlantic_Ocean_beach_plastic_garbage_debris.jpg"),
    ("ghost_net_maldives.jpg",
     "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4b/Marine_organisms_entangled_in_a_ghost_net_within_the_Maldives_%28Olive_Ridley_Project%29.jpg/960px-Marine_organisms_entangled_in_a_ghost_net_within_the_Maldives_%28Olive_Ridley_Project%29.jpg"),
]


def _download_with_ua(url: str, dest: Path) -> bool:
    """Wikimedia-friendly download. Returns True on success."""
    import shutil, urllib.request
    req = urllib.request.Request(url, headers={"User-Agent": WIKIMEDIA_UA})
    try:
        with urllib.request.urlopen(req, timeout=20) as r, dest.open("wb") as f:
            shutil.copyfileobj(r, f)
        return True
    except Exception as exc:
        print(f"[SKIP] Could not fetch {dest.name}: {exc}")
        if dest.exists() and dest.stat().st_size == 0:
            dest.unlink()
        return False


demo_paths = []
for fname, url in FALLBACK_URLS:
    local = DEMO_DIR / fname
    if not local.exists() and not _download_with_ua(url, local):
        continue
    demo_paths.append(local)

print(f"Demo images ready: {len(demo_paths)}")
for p in demo_paths:
    print(f"  - {p}  ({p.stat().st_size // 1024} KiB)")


In [ ]:
def draw_detections(image: Image.Image, detections, title: str = ""):
    """Render `[{box_2d, label, material}, ...]` onto a PIL image. Coordinates are 0-1000 ints."""
    out = image.copy()
    draw = ImageDraw.Draw(out)
    W, H = out.size
    for det in detections or []:
        box = det.get("box_2d") or det.get("box")
        if not box or len(box) < 4:
            continue
        y_min, x_min, y_max, x_max = box
        x1, y1 = int(x_min / 1000 * W), int(y_min / 1000 * H)
        x2, y2 = int(x_max / 1000 * W), int(y_max / 1000 * H)
        draw.rectangle([x1, y1, x2, y2], outline=(0, 212, 255), width=3)
        label = f"{det.get('label', '?')} ({det.get('material', '?')})"
        draw.text((x1 + 4, max(0, y1 - 14)), label, fill=(0, 212, 255))
    if title:
        draw.text((6, 6), title, fill=(255, 255, 255))
    return out

comparison_rows = []
for path in demo_paths[:5]:
    image = Image.open(path).convert("RGB")

    # Base model: temporarily disable the OceanGuard adapter (if loaded).
    if ADAPTER_LOADED:
        model.disable_adapters()
    base_parsed, base_raw, base_t = detect_debris(image)
    if ADAPTER_LOADED:
        model.enable_adapters()

    # Fine-tuned (or base again if adapter missing).
    ft_parsed, ft_raw, ft_t = detect_debris(image)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(draw_detections(image, base_parsed, title="Base Gemma 4 E2B"))
    axes[0].set_axis_off()
    axes[1].imshow(draw_detections(image, ft_parsed, title="OceanGuard fine-tune" if ADAPTER_LOADED else "Base (adapter unavailable)"))
    axes[1].set_axis_off()
    plt.suptitle(path.name)
    plt.tight_layout()
    plt.show()

    comparison_rows.append({
        "image": path.name,
        "base_json_valid": base_parsed is not None,
        "base_n_dets": len(base_parsed) if base_parsed else 0,
        "base_latency_s": round(base_t, 2),
        "ft_json_valid": ft_parsed is not None,
        "ft_n_dets": len(ft_parsed) if ft_parsed else 0,
        "ft_latency_s": round(ft_t, 2),
    })

pd.DataFrame(comparison_rows)

## 6. Quantitative evaluation

The following cell runs a small-scale evaluation against a 50-image COCO-formatted subset of
the held-out validation slice. The complete training+evaluation set lives in the
`OceanguardAI` training repo under `Gemma4-Densea/dataset/` and is
reproducible end-to-end; here we use a downsized subset so the notebook completes inside the
Kaggle wall-clock budget. Metric definitions:

| Metric | Notes |
|---|---|
| `mAP@0.5` | COCO-style mean Average Precision at IoU 0.5, evaluated on the open-vocabulary 50-class taxonomy collapsed to the 8 base classes used by the Android app. |
| `json_valid_rate` | Fraction of generations that produced a parseable JSON array. |
| `label_in_taxonomy_rate` | Fraction of detected labels that map to a known `DebrisType` without falling through to `PLASTIC_DEBRIS` catch-all. |
| `material_attribution_rate` | Fraction of detections where the `material` field was emitted and matched the inferred material from the label. |

Concrete numbers will be filled in after the full training+evaluation run on the RTX 5090. The
placeholders below are intentional (`[mAP_base]`, `[mAP_ft]`, ...) so reviewers can see the
evaluation harness without us inventing metrics.


In [ ]:
# NOTE: This cell expects the eval subset COCO JSON to be mounted at /kaggle/input/oceanguard-eval-50.
# If it's not present, we skip and print the placeholder summary table.
EVAL_ROOT = Path("/kaggle/input/oceanguard-eval-50")
EVAL_COCO = EVAL_ROOT / "annotations.json"

summary_placeholder = pd.DataFrame([
    {"metric": "mAP@0.5",                    "base": "[mAP_base]",        "fine_tuned": "[mAP_ft]"},
    {"metric": "json_valid_rate",            "base": "[json_valid_base]", "fine_tuned": "[json_valid_ft]"},
    {"metric": "label_in_taxonomy_rate",     "base": "[label_tax_base]",  "fine_tuned": "[label_tax_ft]"},
    {"metric": "material_attribution_rate",  "base": "[mat_attr_base]",   "fine_tuned": "[mat_attr_ft]"},
    {"metric": "hallucinated_class_rate",    "base": "[halluc_base]",     "fine_tuned": "[halluc_ft]"},
])

if EVAL_COCO.exists():
    # Full COCO-eval path. Kept compact for review; production runner is scripts/evaluate_detector.py
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    coco_gt = COCO(str(EVAL_COCO))
    image_ids = coco_gt.getImgIds()[:50]
    print(f"[INFO] Evaluating on {len(image_ids)} held-out images from {EVAL_COCO}.")
    # The full conversion (Gemma `box_2d` -> COCO predictions JSON) is implemented in the
    # companion training repo; we surface only the call here to keep this notebook focused
    # on the fine-tune story rather than the eval harness mechanics.
    print("[TODO] Plug in the production evaluator from scripts/evaluate_detector.py to fill\n"
          "       the placeholder cells. The harness is identical for base and fine-tuned models;\n"
          "       only the `model.set_adapter('oceanguard')` toggle differs.")
else:
    print(f"[INFO] Eval subset not mounted at {EVAL_ROOT}. Showing placeholder summary.")

summary_placeholder

## 7. Where the fine-tune sits in the OceanGuard pipeline

The OceanGuard Android pipeline is centred on Gemma 4 E2B served via LiteRT-LM 0.11.0 on the
Galaxy S22 Ultra (Exynos 2200, Xclipse 920 / Vulkan). The Unsloth fine-tune slots in at the
**detection** stage of that pipeline; the rest of the application — tool-calling orchestration,
report writer, Room DB persistence, MarineDex, PDF export — is unchanged by the LoRA.

```text
                ┌────────────────────┐
                │  Camera / Gallery  │
                └─────────┬──────────┘
                          │
                          ▼
            ┌─────────────────────────────────────────┐
            │   Gemma 4 E2B (LiteRT-LM 0.11.0)        │
            │  ┌───────────────────────────────────┐  │
            │  │ Open-vocab detector               │  │
            │  │   box_2d + label + material       │  │
            │  │ ◄── OceanGuard LoRA (this nb)    │  │
            │  └───────────────────────────────────┘  │
            │  ┌───────────────────────────────────┐  │
            │  │ Two-phase tool calling            │  │
            │  │   PHASE 1: tool dispatch          │  │
            │  │   PHASE 2: grounded writer        │  │
            │  └───────────────────────────────────┘  │
            └─────────────────────┬───────────────────┘
                                  │
                                  ▼
                Room DB · MarineDex · PDF / Markdown report
```

| Stage | Component | Fine-tune effect |
|---|---|---|
| Detection | Gemma 4 E2B + OceanGuard LoRA | Taxonomy alignment, JSON validity, material attribution. |
| Tool-call dispatch (PHASE 1) | Gemma 4 E2B + Kotlin tools | None — the fine-tune does **not** rewrite tool-calling syntax to avoid regression on the function-calling head. |
| Report writer (PHASE 2) | Gemma 4 E2B (text-only) | Indirect: better taxonomy reduces upstream noise reaching the report writer. |


## 8. Training procedure summary

The full training script lives in the public training repo at
`OceanguardAI/finetune/train_etapa1.py` and `OceanguardAI/finetune/train_etapa2.py`. This
notebook **does not** execute training (Kaggle T4 is too slow for the full 5-epoch run); it
consumes the resulting adapter. The summary below documents what the published checkpoint
represents.

### Dataset

| Source | Format | Annotation | Used in |
|---|---|---|---|
| CleanSea | COCO JSON | bbox + 8-class | Both stages |
| Ocean_garbage | COCO JSON | bbox + 8-class | Both stages |
| Neural_Ocean | COCO JSON | bbox + 8-class | Both stages |
| **Merged training set** | COCO JSON | bbox + 50 fine-grained labels (mapped from 8 base classes via taxonomy) | Stage 1 |
| **Granular subset (~449 imgs)** | COCO JSON | bbox + 19 material-tipo classes | Stage 2 |
| **VQA mix (10%)** | conversational | text-image pairs, marine domain | Stage 2 (anti-forgetting) |

Total: ~10,247 training images / held-out val / held-out test split.

### Two-stage Unsloth recipe

| Stage | Target | LoRA rank | Modules | Epochs | LR | Notes |
|---|---|---|---|---|---|---|
| 1 | Text-side taxonomy + JSON contract | `r=16, alpha=32` | All linears in language head | 2 | `1e-4` cosine | Vision tower frozen. Focus on `box_2d`/`label`/`material` formatting. |
| 2 | Vision-grounded material attribution | `r=8, alpha=16` | Vision tower + cross-attn | 3 | `1e-4` cosine | 10% VQA mix to prevent catastrophic forgetting of the general visual prior. |

### Hardware

- Training host: RTX 5090 (32 GiB GDDR7), bf16, no quantization at train time.
- Approx. wall-clock: stage 1 `~[stage1_hours]` hours; stage 2 `~[stage2_hours]` hours (filled
  in post-run).
- Effective batch size: `[batch_size]` (gradient-accumulation friendly via Unsloth).

### Reproduction

```bash
# In the OceanguardAI training repo
git clone https://github.com/asferrer/OceanguardAI-App
cd OceanguardAI
python -m venv .venv && source .venv/bin/activate
pip install -r finetune/requirements.txt
python finetune/train_etapa1.py --config finetune/configs/etapa1.yaml
python finetune/train_etapa2.py --config finetune/configs/etapa2.yaml \
    --resume-adapter finetune/outputs/lora_etapa1/adapter
```

## 9. Deployment story (honest)

The shipped Android APK at submission time uses the **base** `google/gemma-4-E2B-it` model,
served via LiteRT-LM 0.11.0 on the Galaxy S22 Ultra (Exynos 2200, Xclipse 920 / Vulkan).
The OceanGuard fine-tune does **not** currently run on-device because the conversion path from
a PEFT LoRA adapter to a `.litertlm` container is blocked on upstream work in Google AI Edge
(`ai-edge-torch` PEFT-export support is in progress as of May 2026).

What this notebook delivers:

1. A fully reproducible **CUDA/Transformers** path that produces the exact same detection
   contract the Android app consumes (`box_2d` + `label` + `material`).
2. A published HuggingFace adapter that any practitioner can drop into Unsloth or stock PEFT.
3. A documented bridge for when on-device LoRA loading becomes available: the same adapter
   converts through `ai-edge-torch`'s PEFT export head once the upstream PR
   (`ai-edge-torch#peft-export`) lands, with no retraining required.

Path to deploy on Android:

- Merge adapter into base model (`peft_model.merge_and_unload()`).
- Re-export through `ai-edge-torch.generative` to a `.litertlm` container (track issue
  upstream).
- Sideload onto device under `app-private/files/models/` and point `VlmModelManager` at the
  fine-tuned checkpoint name (`gemma-4-E2B-it-oceanguard.litertlm`).

Risk acknowledged in the submission writeup: the on-device fine-tune is **not expected** to
ship in the APK by the 18 May 2026 hackathon deadline. The fine-tune itself, the adapter, and
this reproducibility notebook are the artifacts targeted for the Unsloth bonus track.

## 10. Conclusion, future work, and references

**Conclusion.** A targeted two-stage Unsloth fine-tune of Gemma 4 E2B is a tractable way to
harden an open-vocabulary VLM around a specialised taxonomy without sacrificing the JSON
structure that downstream tool-calling depends on. The LoRA adapter is small enough
(`~50 MB`) to distribute on HuggingFace and to (once upstream tooling lands) merge into the
shipped `.litertlm` for on-device deployment.

**Future work.**

1. **On-device LoRA application.** Track `ai-edge-torch` PEFT-export support and ship the
   merged checkpoint as `gemma-4-E2B-it-oceanguard.litertlm`.
2. **Calibration of confidence scores.** Replace the placeholder `0.75` confidence on Gemma 4
   detections with per-token log-probability aggregates once LiteRT-LM exposes them.
3. **Multilingual instruction tuning.** Add a fourth stage that trains the model to obey the
   six on-device report languages without translation drift.
4. **Mask supervision.** Extend the dataset with segmentation masks to unlock the planned
   RF-DETR-Seg pipeline.

**References.**

- Daniel Han et al. *Unsloth — 2-5x faster fine-tuning with 80% less memory.* https://github.com/unslothai/unsloth
- Google DeepMind. *Gemma 4 Technical Report.* 2026.
- Hu et al. *LoRA: Low-Rank Adaptation of Large Language Models.* arXiv:2106.09685.
- Mangrich et al. *CleanSea: A dataset for marine debris detection.* 2022.


## 11. Reproducibility checklist

1. **Clone the training repo.**
   ```bash
   git clone https://github.com/asferrer/OceanguardAI-App
   cd OceanguardAI
   ```
2. **Set up environment.** Python 3.11, CUDA 12.1+, PyTorch 2.4+. Use the `finetune/requirements.txt`.
3. **Download datasets.** CleanSea + Ocean_garbage + Neural_Ocean COCO JSONs into
   `finetune/datasets/`. Original license terms (CC-BY / equivalent)
   apply to the source datasets; the OceanGuard merge script is Apache 2.0.
4. **Stage 1 training.** `python finetune/train_etapa1.py --config finetune/configs/etapa1.yaml`.
5. **Stage 2 training.** `python finetune/train_etapa2.py --config finetune/configs/etapa2.yaml --resume-adapter finetune/outputs/lora_etapa1/adapter`.
6. **Evaluation.** `python scripts/evaluate_detector.py --adapter finetune/outputs/lora_etapa2/adapter --eval-set docs/submission/eval_set/`.
7. **Publish adapter.** Follow `docs/submission/upload_to_hf.md`.
8. **Reproduce this notebook.** Attach the published adapter as a dataset in Kaggle and Run All.

Expected wall-clock end-to-end on an RTX 5090: ~`[total_hours]` hours including data prep.
Expected disk footprint: `[disk_gb]` GB.


## Submission Metadata

- **Track**: Global Resilience · Unsloth $10K Bonus
- **License**: Apache 2.0 (LoRA adapter) · Gemma Terms of Use (base model)
- **Source repo**: https://github.com/asferrer/OceanguardAI-App
- **HuggingFace adapter**: https://huggingface.co/asferrer/gemma-4-E2B-it-oceanguard-marine-debris
- **APK download**: https://github.com/asferrer/OceanguardAI/releases/latest
- **Companion notebook (base pipeline)**: `docs/submission/notebook.ipynb`
- **Submission writeup**: `docs/submission/WRITEUP_FINAL.md`
- **Author**: Alejandro Sanchez Ferrer (`asanc.tech@gmail.com`)